<a href="https://colab.research.google.com/github/irenerope/intro-machine-learning-language-processing/blob/main/Bonus_Exercise_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Bonus Exercises 1: Aligning Multilingual Embedding Spaces**



This notebook represents the first bonus exercises for the lecture Multilingual and Crosslingual Methods and Language Resources (2026W). For each successfully completed bonus exercise, a maximum of three points can be achieved that will be added to the points of the final exam. The tasks to be completed in the following notebook are marked with 👋 ⚒.



In this notebook, you will perform and evaluate a supervised method for aligning the embedding spaces of two languages. The examples in the notebook rely on the language pair English-German, however, feel free to change this pair to languages of your choice from the available embeddings and dictionaries (see below).

-----------
## **Preparing the Embeddings and Data**

In this notebook, we will be using fastText embeddings that represents a character-based version of the word2vec skipgram method. Details on the method can be found in the [original publication](https://aclanthology.org/Q17-1010.pdf) and [this website](https://fasttext.cc/).

Pretrained fastText embeddings are available in [157 languages](https://fasttext.cc/docs/en/crawl-vectors.html). The following code cell loads the fastText embeddings for English and German.

👋 ⚒ Please change the following download command if you wish to align other languages than English and German.

Before you decide on a final language pair, please make sure that:
1.   There are pretrained embeddings for this language (see [here](https://fasttext.cc/docs/en/crawl-vectors.html))
2.   There is a bilingual word list available (see the [MUSE GitHub](https://github.com/facebookresearch/MUSE/tree/main) section "Ground-truth bilingual dictionaries")

If the embeddings are available, change the two-digit ISO code in `cc.en.300.vec.g` and `cc.de.300.vec.gz` to the language(s) of your choice, e.g. sk for Slovak.

In [1]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz  # English
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.de.300.vec.gz  # German

--2026-11-04 19:51:51--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 54.192.138.127, 54.192.138.98, 54.192.138.13, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|54.192.138.127|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1325960915 (1.2G) [binary/octet-stream]
Saving to: ‘cc.en.300.vec.gz’

cc.en.300.vec.gz    100%[===================>]   1.23G   170MB/s    in 9.7s    

2026-11-04 19:52:00 (131 MB/s) - ‘cc.en.300.vec.gz’ saved [1325960915/1325960915]

--2026-11-04 19:52:00--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.de.300.vec.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 54.192.138.127, 54.192.138.98, 54.192.138.13, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|54.192.138.127|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1278030050 (1.2G) [binary/octet-stream]
Saving to

### Loading the Embeddings

As a next step we will unzip and load the embeddings. For this alignment task, we will only use the top 100,000 words for both languages to speed up the processing. This choice of only using the top 100,000 words also depends on the lenght of the available bilingual word lists.

In [14]:
import gzip
import numpy as np

def load_fasttext_embeddings(file_path, top_n):
    embeddings = {}
    with gzip.open(file_path, 'rb') as f:
        for i, line in enumerate(f):
            # Line 0 is a header line
            if i > 0 and i <= top_n:
              tokens = line.decode('utf-8').strip().split(' ')
              word = tokens[0]
              vector = np.array(tokens[1:], dtype=np.float32)
              vector = vector / np.linalg.norm(vector)
              embeddings[word] = vector
    return embeddings

# Load the top English and German embeddings for the top 100,000 words (100000)
# FastText sorts the embeddings by decreasing order of word frequency by default
en_embeddings = load_fasttext_embeddings('cc.en.300.vec.gz', 100000)
de_embeddings = load_fasttext_embeddings('cc.de.300.vec.gz', 100000)

print(f"Loaded {len(en_embeddings)} English embeddings")
print(f"Loaded {len(de_embeddings)} German embeddings")


Loaded 100000 English embeddings
Loaded 100000 German embeddings


Let us explore the format of the downloaded and loaded embeddings.

In [6]:
print(f'The loaded embeddings represent a {type(en_embeddings)} datatype.\n')
print(f'Each entry represents the word and the related embedding.\n')
print(f'We can query the word as a key and obtain the embedding, e.g. for good the embedding is {en_embeddings["good"]}.\n')
print(f'The dimensionality of these embeddings corresponds to {len(en_embeddings["good"])}.')

The loaded embeddings represent a <class 'dict'> datatype.

Each entry represents the word and the related embedding.

We can query the word as a key and obtain the embedding, e.g. for good the embedding is [-0.08404064 -0.05785208  0.00155124  0.1233691  -0.05985956  0.00565746
  0.11506542 -0.01505614  0.01587738 -0.00118624 -0.0886031   0.02126109
  0.00912493  0.00419747  0.01450865  0.0062962   0.07829193 -0.01815862
 -0.0549321  -0.02126109  0.01076742  0.07500695  0.01359615  0.00821244
  0.00638745 -0.05867332  0.03056853 -0.01916236  0.0617758   0.0275573
  0.06569952 -0.05192087 -0.03987596  0.00583996  0.04005846  0.05520585
 -0.00556621 -0.11187168 -0.03221102 -0.02463732 -0.01879736  0.0068437
 -0.0062962   0.03312351 -0.03020353  0.05292461  0.00757369 -0.05785208
 -0.05274212  0.00994618 -0.08440564  0.0142349  -0.03722973  0.00611371
 -0.05812582  0.05365461  0.06579077 -0.04918339 -0.13377152 -0.03695598
 -0.02290358 -0.04516842 -0.04763215 -0.06250579  0.04261344  0.0

### Downloading and Loading the Bilingual Word List

To perform this alignment, we will use a bilingual word list that is provided by the Multilingual Unsupervised and Supervised Embeddings (MUSE) project (see [here](https://github.com/facebookresearch/MUSE/tree/main) for all languages).

👋 ⚒ Please change the following downloading command to the language pair of your choice (as long as available on MUSE).


In [7]:
!wget https://dl.fbaipublicfiles.com/arrival/dictionaries/en-de.txt

--2026-11-04 20:07:34--  https://dl.fbaipublicfiles.com/arrival/dictionaries/en-de.txt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 54.192.138.44, 54.192.138.127, 54.192.138.13, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|54.192.138.44|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1742131 (1.7M) [text/x-c++]
Saving to: ‘en-de.txt.1’

en-de.txt.1         100%[===================>]   1.66M  --.-KB/s    in 0.03s   

2026-11-04 20:07:34 (65.8 MB/s) - ‘en-de.txt.1’ saved [1742131/1742131]



### Creating a bilingual word list

As a next step, we will create a bilingual word list from the donwloaded text file.

👋 ⚒ Create a list of tuples `[(en_word1, de_word1), (en_word2, de_word2),...]`from the downloaded text file in the following code cell. To complete this task, please complement the provided function `load_bilingual_word_list` where it says `Your code here`.

For English-German, the first ten tuples of the list look like this:

```
[('the', 'die'), ('the', 'der'), ('the', 'dem'), ('the', 'den'), ('the', 'das'), ('and', 'sowie'), ('and', 'und'), ('was', 'war'), ('was', 'wurde'), ('for', 'für')]
```



In [8]:
'''
Create a list of tuples that contain word translations

Parameters:
Text file with one bilingual word pair per line

Returns:
A list of tuples that each contains one bilingual word pair
'''
def load_bilingual_word_list(file_path):
    bilingual_list = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            if len(parts) == 2:
                bilingual_list.append((parts[0], parts[1]))
    return bilingual_list

# Load English-German word pairs
en_de_pairs = load_bilingual_word_list('en-de.txt')

print(en_de_pairs[:10])

[('the', 'die'), ('the', 'der'), ('the', 'dem'), ('the', 'den'), ('the', 'das'), ('and', 'sowie'), ('and', 'und'), ('was', 'war'), ('was', 'wurde'), ('for', 'für')]


### Getting the Embeddings for our Word List

As a next step, we need to see which words from the word list have a vector representation in the embedding space for both languages and create a list of corresponding embeddings for both languages.


In [9]:
import numpy as np

'''
Function to create a list of word embeddings that is parallel to a bilingual list of words

Parameters:
Bilingual list of words, embeddings in the first language, embeddings in the second language

Returns:
Two numpy arrays of embeddings that correspond two the bilingual word list
'''
def extract_word_embeddings(bilingual_pairs, en_embeddings, de_embeddings):
    en_vecs = []
    de_vecs = []

    for en_word, de_word in bilingual_pairs:
        if en_word in en_embeddings and de_word in de_embeddings:
            en_vecs.append(en_embeddings[en_word])
            de_vecs.append(de_embeddings[de_word])

    # Convert lists to numpy arrays
    en_vecs = np.array(en_vecs)
    de_vecs = np.array(de_vecs)

    return en_vecs, de_vecs

# Extract English and German embeddings for the bilingual lexicon
en_vecs, de_vecs = extract_word_embeddings(en_de_pairs, en_embeddings, de_embeddings)

print(f"Extracted {en_vecs.shape[0]} aligned word vectors.")

Extracted 22546 aligned word vectors.


-----------
## **Embedding Alignment**

We will now use the dictionary and embeddings to align the two vector spaces. The English vector space will be aligned to the German vector space using the [Procrustes](https://en.wikipedia.org/wiki/Orthogonal_Procrustes_problem) alignment method.

Given two matrices, Procrustes finds an orthogonal matrix which most closely maps one input matrix to the other. As a first step, we need to compute this orthogonal transformation matrix.  



In [10]:
"""
Function to perform orthogonal Procrustes alignment to learn a mapping from X to Y.

Parameters:
X (numpy array): Source language word embeddings (English)
Y (numpy array): Target language word embeddings (German)

Returns:
W (numpy array): Orthogonal transformation matrix
"""
def orthogonal_procrustes(X, Y):
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    Y = Y / np.linalg.norm(Y, axis=1, keepdims=True)

    # Compute matrix product of X^T and Y
    M = np.dot(X.T, Y)

    # Perform Singular Value Decomposition (SVD) on the matrix M
    U, _, Vt = np.linalg.svd(M)

    # Compute the orthogonal transformation matrix W
    W = np.dot(U, Vt)

    return W

W = orthogonal_procrustes(en_vecs, de_vecs)

print("Orthogonal mapping matrix learned.")


Orthogonal mapping matrix learned.


In a second step, the obtained matrix is used to learn an orthogonal mapping of the English vector space to approximate it to the German vector space. Here we can transform the entire vector space of 100,000 embeddings.

In [11]:
"""
Apply the learned orthogonal mapping to the source language embeddings.

Parameters:
embeddings (dict): Source language embeddings (English)
W (numpy array): Orthogonal transformation matrix

Returns:
mapped_embeddings (dict): Transformed embeddings
"""
def apply_mapping(embeddings, W):
    mapped_embeddings = {}
    for word, vec in embeddings.items():
        mapped_vec = np.dot(vec, W)
        # Normalize the mapped vector
        mapped_vec = mapped_vec / np.linalg.norm(mapped_vec)
        mapped_embeddings[word] = mapped_vec
    return mapped_embeddings

aligned_en_embeddings = apply_mapping(en_embeddings, W)

print(f"Aligned {len(aligned_en_embeddings)} English embeddings with the German embedding space.")


Aligned 100000 English embeddings with the German embedding space.


-----------
## **Evaluation**

In this part, you will explore two different tasks for evaluating the final vector space:


1.   Word Translation
2.   Cross-Lingual Analogy Completion



### Word Translation

We will now use the bilingual word list downloaded from MUSE to evaluate the ability of our newly created aligned embedding space to translate words from English to German.

A function that takes an English word as input and ouputs the nearest neighors of the German vector space is already provided for your convenience.

In [12]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_nn(word, aligned_en_embeddings, de_embeddings, top_k):
    print("Nearest neighbors of \"%s\":" % word)
    en_vec = aligned_en_embeddings[word]
    de_words = list(de_embeddings.keys())
    de_vecs = np.array(list(de_embeddings.values()))

    # Compute cosine similarity between the English word vector and all German word vectors
    en_vec = en_vec / np.linalg.norm(en_vec)
    de_vecs_norm = de_vecs / np.linalg.norm(de_vecs, axis=1, keepdims=True)
    similarities = cosine_similarity([en_vec], de_vecs_norm).flatten()

    # Get top_k most similar German words
    nearest_idxs = similarities.argsort()[-top_k:][::-1]
    nearest_words = [de_words[i] for i in nearest_idxs]

    return nearest_words

en_word = 'the'
nearest_neighbors = get_nn(en_word, aligned_en_embeddings, de_embeddings, 5)
print(nearest_neighbors)

Nearest neighbors of "the":
['der', 'die', 'den', 'dem', 'besagten']


👋 ⚒ Use the already downloaded bilingual word list to evaluate the ability of our aligned vector space to translate from the first language to the second. The output of this task should be the **accuracy** calculated on the first **1000 words** from the MUSE word list. Calculate the accuracy by counting how often the correct translation from the list is contained in the five nearest neighbors returned by the function `get_nn` in the other language and divide that number by the total number of translation attempts, i.e., 1000.

Use the provided function `get_nn` to obtain the *k* nearest words in the vector space in German, given an English input word.


In [ ]:
correct = 0
total = 0

for en_word, de_word in en_de_pairs[:1000]:
    total += 1
    if en_word in aligned_en_embeddings:
        neighbors = get_nn(en_word, aligned_en_embeddings, de_embeddings, 5)
        if de_word in neighbors:
            correct += 1

accuracy = correct / total
print(f"Correct: {correct} / {total}")
print(f"Accuracy (top-5): {accuracy:.2%}")

Nearest neighbors of "the":
Nearest neighbors of "the":
Nearest neighbors of "the":
Nearest neighbors of "the":
Nearest neighbors of "the":
Nearest neighbors of "and":
Nearest neighbors of "and":
Nearest neighbors of "was":
Nearest neighbors of "was":
Nearest neighbors of "for":
Nearest neighbors of "that":
Nearest neighbors of "that":
Nearest neighbors of "with":
Nearest neighbors of "from":
Nearest neighbors of "from":
Nearest neighbors of "from":
Nearest neighbors of "from":
Nearest neighbors of "this":
Nearest neighbors of "this":
Nearest neighbors of "this":
Nearest neighbors of "his":
Nearest neighbors of "his":
Nearest neighbors of "his":
Nearest neighbors of "his":
Nearest neighbors of "his":
Nearest neighbors of "not":
Nearest neighbors of "not":
Nearest neighbors of "not":
Nearest neighbors of "are":
Nearest neighbors of "talk":
Nearest neighbors of "talk":
Nearest neighbors of "talk":
Nearest neighbors of "talk":
Nearest neighbors of "which":
Nearest neighbors of "which":
Ne

### Cross-Lingual Analogy Completion

An analogy compares two related pairs of words, e.g. *man is to woman as king is to queen*. This task can be extended to use analogies for translation, e.g. *man is to woman as Mann ist zu Frau*.


👋 ⚒ Create **twenty** examples of crosslingual analogies and see whether the aligned vector space is able to correctly complete analogies across languages, e.g. positive=(queen, König), negative=(king). You can use examples from the analogy text file in GitHub for this purpose.

Hints:


*   Multilingual Analogies: To create the examples, all you need is a translation of an existing analogy. You can use the already loaded bilingual word list to obtain the translations and the existing analogy list (anlogies.txt on Github) to obtain analogies.
*   Implementation: In the code below, the word to obtain `vec_c`needs to be a translation of the word for `vec_a` and you need to search in the embedding space of the other language, e.g. `de_embeddings` in this notebook example.

In [16]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def norm(vec):
  return vec / np.linalg.norm(vec)

def get_target_words(embeddings, vec_a, vec_b, vec_c, top_k):
    words = list(embeddings.keys())
    vecs = np.array(list(embeddings.values()))

    # Compute analogy based on input vectors b+c-a (woman+king-man)
    positive = norm(vec_b+vec_c)
    target_vec = norm(positive - vec_a)
    vecs_norm = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    similarities = cosine_similarity([target_vec], vecs_norm).flatten()

    # Get top_k most similar words for the retrieved result vector d
    nearest_idxs = similarities.argsort()[-top_k:][::-1]
    nearest_words = [words[i] for i in nearest_idxs]

    return nearest_words

# Example English analogy
vec_a = norm(aligned_en_embeddings["man"])
vec_b = norm(aligned_en_embeddings["woman"])
vec_c = norm(aligned_en_embeddings["king"])

nearest_neighbors = get_target_words(aligned_en_embeddings, vec_a, vec_b, vec_c, 5)
print(nearest_neighbors)

# Example cross-lingual analogy
vec_a_en = norm(aligned_en_embeddings["king"])
vec_b_en = norm(aligned_en_embeddings["queen"])
vec_c_de = norm(de_embeddings["König"])

nearest_neighbors_cross = get_target_words(de_embeddings, vec_a_en, vec_b_en, vec_c_de, 5)
print(nearest_neighbors_cross)

['queen', 'king', 'Queen', 'queens', 'royal']
['Königin', 'Königinnen', 'Unternehmerin', 'König', 'Gewinnerin']


In [ ]:
analogies = [
    # Gender / family
    ("king", "queen", "König", "Königin"),
    ("man", "woman", "Mann", "Frau"),
    ("father", "mother", "Vater", "Mutter"),
    ("brother", "sister", "Bruder", "Schwester"),
    ("son", "daughter", "Sohn", "Tochter"),
    ("boy", "girl", "Junge", "Mädchen"),
    ("uncle", "aunt", "Onkel", "Tante"),
    ("grandfather", "grandmother", "Großvater", "Großmutter"),
    # Capital : country
    ("Rome", "Italy", "Rom", "Italien"),
    ("Madrid", "Spain", "Madrid", "Spanien"),
    ("Vienna", "Austria", "Wien", "Österreich"),
    ("Paris", "France", "Paris", "Frankreich"),
    # Comparative
    ("good", "better", "gut", "besser"),
    ("big", "bigger", "groß", "größer"),
    ("small", "smaller", "klein", "kleiner"),
    # Past tense
    ("go", "went", "gehen", "ging"),
    ("take", "took", "nehmen", "nahm"),
    # Plural
    ("dog", "dogs", "Hund", "Hunde"),
    ("car", "cars", "Auto", "Autos"),
    ("child", "children", "Kind", "Kinder"),
]

correct = 0

for a_en, b_en, c_de, expected in analogies:
    if a_en not in aligned_en_embeddings or b_en not in aligned_en_embeddings or c_de not in de_embeddings:
        print(f"Skipped (word missing): {a_en}, {b_en}, {c_de}")
        continue

    vec_a = norm(aligned_en_embeddings[a_en])
    vec_b = norm(aligned_en_embeddings[b_en])
    vec_c = norm(de_embeddings[c_de])

    result = get_target_words(de_embeddings, vec_a, vec_b, vec_c, 5)

    if expected in result:
        correct += 1
        mark = "V"
    else:
        mark = "X"

    print(f"{a_en} : {b_en} = {c_de} : ?  →  {result}  {mark} (expected: {expected})")

print(f"\nCorrect: {correct} / {len(analogies)}")